# Joint-Digital MIT — Upgrade Pipeline (Steps 1 & 2)
**Time-varying AI series + submarine-cable instrument → re-run headline & IV**

Run top to bottom. You'll upload your panel in Step 0. The two download cells need
open internet (Colab has it). If a download 404s, the cell tells you exactly which
file to upload as a fallback.

What this fixes: your current AI index is time-invariant within country, so fixed-effects
estimates are attenuated and neither positives nor nulls are interpretable. This adds a
within-country-*varying* AI measure and an *external* instrument (cable arrival), which is
what makes the findings credible.

In [ ]:
# 1. Setup
!pip install -q linearmodels country_converter pandas numpy
import io, json, urllib.request, numpy as np, pandas as pd, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("country_converter").setLevel(logging.ERROR)
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
import country_converter as coco
_CC = coco.CountryConverter()
def to_iso3(names):
    r = _CC.convert(list(map(str, names)), to="ISO3", not_found=None)
    if isinstance(r, str): r=[r]
    return [None if (isinstance(x,list) or x=="not found") else x for x in r]
def _get(url, t=60):
    req=urllib.request.Request(url, headers={"User-Agent":"academic/1.0"})
    return urllib.request.urlopen(req, timeout=t).read()
print("setup done")

## Step 0 — Upload your files
Click **Choose Files** and upload:
- **your panel** (e.g. `final_panel_imputed__5_.csv`) — required
- *optional fallbacks* if the downloads fail: `ai_timevarying.csv` (cols: country, year, ai_adopt) and/or `cables.json`

You can also use Google Drive instead (see the commented alternative below).

In [ ]:
# Upload files (Colab file picker)
from google.colab import files
uploaded = files.upload()           # pick your panel CSV (+ optional fallbacks)
print("uploaded:", list(uploaded.keys()))

# ---- ALTERNATIVE: Google Drive (uncomment to use instead of upload) ----
# from google.colab import drive; drive.mount('/content/drive')
# PANEL_PATH = '/content/drive/MyDrive/fintech_ai_mit/data/processed/final_panel_imputed__5_.csv'

# auto-detect the panel = the largest uploaded .csv whose columns include iso3,year,F
PANEL_PATH = None
for fn in uploaded:
    if fn.lower().endswith('.csv'):
        try:
            head = pd.read_csv(fn, nrows=1)
            if {'iso3','year'}.issubset(head.columns):
                PANEL_PATH = fn
        except Exception:
            pass
PANEL_PATH = PANEL_PATH or next((f for f in uploaded if f.endswith('.csv')), None)
print("panel ->", PANEL_PATH)
assert PANEL_PATH, "No panel CSV detected — re-run and upload your panel."
panel = pd.read_csv(PANEL_PATH, low_memory=False).sort_values(['iso3','year'])
print("panel:", panel.shape, "| countries:", panel['iso3'].nunique())

## Step 1 — Time-varying AI adoption series
Primary source: **OECD.AI** AI venture-capital investment by country-year. If the link has
rotated, download the CSV from https://oecd.ai/en/data (AI investments → download), save it
as `ai_timevarying.csv` with columns `country,year,ai_adopt`, and re-upload in Step 0.

In [ ]:
OECD_VC_CSV = "https://oecd.ai/assets/data/ai-investments-by-country.csv"  # verify/replace if 404
AI_COL = None
ai = None
try:
    ai = pd.read_csv(io.BytesIO(_get(OECD_VC_CSV)))
    ai.columns=[c.lower().strip() for c in ai.columns]
    cc=next(c for c in ai.columns if "country" in c)
    yc=next(c for c in ai.columns if "year" in c)
    vc=next(c for c in ai.columns if c in ("value","investment","amount","usd","ai_adopt"))
    ai=ai.rename(columns={cc:"country",yc:"year",vc:"ai_adopt"})[["country","year","ai_adopt"]].dropna()
    print("OECD.AI series:", ai.shape)
except Exception as e:
    print("OECD download failed:", str(e)[:90])
    import os
    if os.path.exists("ai_timevarying.csv"):
        ai = pd.read_csv("ai_timevarying.csv"); print("using uploaded ai_timevarying.csv:", ai.shape)
    else:
        print("No AI series -> will fall back to ai_pubs_log already in your panel.")

if ai is not None:
    ai["iso3"]=to_iso3(ai["country"]); ai=ai.dropna(subset=["iso3"])
    ai["year"]=pd.to_numeric(ai["year"],errors="coerce")
    ai=ai.groupby(["iso3","year"],as_index=False)["ai_adopt"].sum()
    ai["ai_adopt_log"]=np.log1p(ai["ai_adopt"].clip(lower=0))
    panel=panel.drop(columns=[c for c in ["ai_adopt_log"] if c in panel.columns])
    panel=panel.merge(ai[["iso3","year","ai_adopt_log"]],on=["iso3","year"],how="left")
    AI_COL="ai_adopt_log"
    print("within-country SD of new AI series:",
          round(panel.groupby('iso3')['ai_adopt_log'].std().median(),3))
else:
    AI_COL="ai_pubs_log"
print("AI column used for G:", AI_COL)

## Step 2 — Submarine-cable instrument
TeleGeography's public cable API gives each cable's ready-for-service year and landing
countries (the Hjort–Poulsen 2019 digital-infrastructure instrument). If the API is
unreachable, download `cable/all.json` from submarinecablemap.com (or the Hjort–Poulsen
openICPSR cable file), save as `cables.json`, and re-upload.

In [ ]:
CAND=["https://www.submarinecablemap.com/api/v3/cable/all.json",
      "https://raw.githubusercontent.com/telegeography/www.submarinecablemap.com/master/web/public/api/v3/cable/all.json"]
cables=None
for u in CAND:
    try: cables=json.loads(_get(u)); break
    except Exception: pass
import os
if cables is None and os.path.exists("cables.json"):
    cables=json.load(open("cables.json"))
HAVE_IV=False
if cables:
    rows=[]
    for c in cables:
        cid=c.get("id"); rfs=str(c.get("rfs",""))[:4]
        try: yr=int(rfs)
        except: continue
        try:
            det=json.loads(_get(f"https://www.submarinecablemap.com/api/v3/cable/{cid}.json"))
            lps=det.get("landing_points",det.get("landings",[]))
            ccs={lp.get("country") for lp in lps if isinstance(lp,dict)}
        except Exception: ccs=set()
        for country in ccs:
            if country: rows.append((country,yr))
    cab=pd.DataFrame(rows,columns=["country","rfs_year"]).dropna()
    cab["iso3"]=to_iso3(cab["country"]); cab=cab.dropna(subset=["iso3"])
    grid=panel[["iso3","year"]].drop_duplicates()
    firsts=cab.groupby("iso3")["rfs_year"].min().rename("first_cable_year")
    grid=grid.merge(firsts,on="iso3",how="left")
    grid["cables_cum"]=[int((cab[cab["iso3"]==i]["rfs_year"]<=y).sum()) for i,y in zip(grid["iso3"],grid["year"])]
    panel=panel.drop(columns=[c for c in ["cables_cum","first_cable_year"] if c in panel.columns])
    panel=panel.merge(grid[["iso3","year","cables_cum","first_cable_year"]],on=["iso3","year"],how="left")
    HAVE_IV=True
    print("cable instrument merged; countries with >=1 cable:",
          panel['cables_cum'].gt(0).groupby(panel['iso3']).any().sum())
else:
    print("No cable data available — IV cell will be skipped. Upload cables.json to enable.")

## Step 3 — Rebuild time-varying G, FG, quadrant terms, lags

In [ ]:
panel["G_tv"]=panel.groupby("year")[AI_COL].transform(lambda s:(s-s.mean())/s.std())
panel["FG_tv"]=panel["F"]*panel["G_tv"]
panel=panel.sort_values(["iso3","year"])
panel["G_tv_l"]=panel.groupby("iso3")["G_tv"].shift(1)
panel["FG_tv_l"]=panel.groupby("iso3")["FG_tv"].shift(1)
for qd in ["HH","HL","LH","LL"]:
    col=f"FG_l_{qd}"
    if col in panel.columns:
        m=panel[col].fillna(0)!=0
        panel[f"FG_tv_l_{qd}"]=np.where(m,panel["FG_tv_l"],0.0)
if HAVE_IV:
    panel["cables_cum_l"]=panel.groupby("iso3")["cables_cum"].shift(1)
print("G_tv within-country SD =", round(panel.groupby('iso3')['G_tv'].std().median(),3),
      "(should be >0 — that's the whole point)")
panel.to_csv("panel_upgraded.csv",index=False)
print("saved panel_upgraded.csv")

## Step 4 — Re-run the headline & IV with the upgraded data
Watch: (a) does `FG_tv_LH` survive with time-varying G? (b) is the cable-IV first-stage
F above ~10? A strong first stage + surviving coefficient = causal finding; a clean null
with a well-measured, instrumented regressor = a *credible* null (answers the referee).

In [ ]:
Cl=[f"{c}_l" for c in ["log_gdp","trade_pct_gdp","inflation","investment_pct_gdp",
     "pop_growth","credit_to_private_pct_gdp","secondary_enrollment"]]
P=panel.set_index(["iso3","year"])
star=lambda p:"***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""

print("== Baseline with time-varying G ==")
d=P[["gdp_pc_growth","F_l","G_tv_l","FG_tv_l"]+Cl].dropna()
r=PanelOLS.from_formula("gdp_pc_growth ~ F_l+G_tv_l+FG_tv_l+"+" + ".join(Cl)+
    " + EntityEffects + TimeEffects",data=d,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
for k in ["F_l","G_tv_l","FG_tv_l"]:
    print(f"   {k:<10}{r.params[k]:+.3f} (p={r.pvalues[k]:.3f}) {star(r.pvalues[k])}")

print("\n== LH quadrant with time-varying G ==")
q=[f"{v}_l_{qd}" for qd in ["HH","HL","LH","LL"] for v in ["F","G"]]+[f"FG_tv_l_{qd}" for qd in ["HH","HL","LH","LL"]]
q=[c for c in q if c in P.columns]
d=P[["gdp_pc_growth"]+q+Cl].dropna()
r=PanelOLS.from_formula("gdp_pc_growth ~ "+" + ".join(q+Cl)+" + EntityEffects + TimeEffects",
    data=d,drop_absorbed=True).fit(cov_type="clustered",cluster_entity=True)
if "FG_tv_l_LH" in r.params.index:
    print(f"   FG_tv_LH {r.params['FG_tv_l_LH']:+.3f} (p={r.pvalues['FG_tv_l_LH']:.3f}) {star(r.pvalues['FG_tv_l_LH'])}")

if HAVE_IV:
    print("\n== IV: instrument joint-digital with CABLE arrival ==")
    dz=panel.dropna(subset=["gdp_pc_growth","FG_tv_l","cables_cum_l","F_l","G_tv_l"]+Cl)
    Pz=dz.set_index(["iso3","year"])
    try:
        iv=IV2SLS.from_formula("gdp_pc_growth ~ 1 + F_l + G_tv_l + "+" + ".join(Cl)+
            " + [FG_tv_l ~ cables_cum_l]",data=Pz).fit(cov_type="clustered",
            clusters=Pz.index.get_level_values(0))
        print(f"   FG_tv (cable-IV) {iv.params['FG_tv_l']:+.3f} (p={iv.pvalues['FG_tv_l']:.3f})")
        print(f"   first-stage F = {iv.first_stage.diagnostics.iloc[0]['f.stat']:.1f}")
    except Exception as e:
        print("   IV failed:", str(e)[:100])

## Step 5 — Download the upgraded panel

In [ ]:
from google.colab import files
files.download("panel_upgraded.csv")